In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from datetime import datetime, timedelta

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document

In [22]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embedding_size = 1536

index = faiss.IndexFlatL2(embedding_size)

In [23]:
vectorstore = FAISS(
    embedding_function= embeddings,
    index = index,
    docstore= InMemoryDocstore({}),
    index_to_docstore_id= {},
)

In [27]:
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.000000000000000000000000000001,
    k=1,
)

print("TimeWeightedVectorStoreRetriever 준비 완료")

TimeWeightedVectorStoreRetriever 준비 완료


In [29]:
yesterday = datetime.now() - timedelta(days=1)

retriever.add_documents([
    Document(
        page_content="오래된 문서입니다.",
        metadata={"last_accessed_at": yesterday},
    ),
    Document(
        page_content="최신 문서입니다.",
    ),
])

['a4c48b78-65b8-434c-8d51-9d0737920c11',
 '4011fc0a-d524-47cb-92dc-6f984aea0cb5']

In [30]:
results = retriever.invoke("문서")

for doc in results:
    print(doc.page_content)
    print(doc.metadata)

오래된 문서입니다.
{'last_accessed_at': datetime.datetime(2026, 10, 2, 15, 10, 13, 689501), 'created_at': datetime.datetime(2026, 10, 2, 15, 10, 12, 164209), 'buffer_idx': 0}


In [ ]:
from langchain_community.retrievers import BM25Retriever

sample_texts = [
    "자동차 보험 가입 방법과 보험료 계산에 대해 설명합니다.",
    "차량을 운전할 때 필요한 보장과 사고 처리 방법을 설명합니다.",
    "자동차 보험은 운전자에게 중요한 금융 상품입니다.",
    "오늘 날씨가 매우 좋습니다.",
]

# 임베딩을 사용하지 않는 키워드 기반 검색
bm25 = BM25Retriever.from_texts(sample_texts)
bm25.k = 2

results = bm25.invoke("자동차 보험")

for i, doc in enumerate(results):
    print(f"[{i + 1}] {doc.page_content}")

[1] 자동차 보험 가입 방법과 보험료 계산에 대해 설명합니다.
[2] 오늘 날씨가 매우 좋습니다.
